# 02-11. 그룹화 계산(groupby)

## groupby()
: 데이터를 그룹화하여 연산을 수행하는 메서드이다.

### 사용법
```python
df.groupby(by=None, axis=0, level=None, as_index=True, sort=True, group_keys=True, squeeze=NoDefault.no_default, observed=False, dropna=True)
```
- by : 그룹화할 내용이다. 함수, 축, 리스트 등이 올 수 있다.
- axis : 그룹화를 적용할 축이다.
- level : MultiIndex의 경우 레벨을 지정할 수 있다.
- as_index : 그룹화할 내용을 인덱스로 할지 여부이다. False이면 기존 인덱스가 유지된다.
- sort : 그룹 키를 정렬할지 여부이다.
- group_keys : apply 메서드 사용 시 결과에 따라 그룹화 대상인 열이 인덱스와 중복(group key)될 수 있다. 이때 `group_keys=False`로 인덱스를 기본값으로 지정할 수 있다.
- squeeze : 결과가 1행 or 1열짜리 데이터일 경우 Series로, 1행 & 1열짜리 데이터일 경우 스칼라로 출력한다.
- observed : Categorical로 그룹화할 경우 Categorical 그룹퍼에 의해 관찰된 값만 표시할지 여부이다.
- dropna : 결측값을 계산에서 제외할지 여부이다.

In [1]:
import numpy as np
import pandas as pd

In [ ]:
# 15 x 3 난수 DataFrame 생성 (index 열을 그룹 기준으로 사용)
idx = ['A', 'A', 'B', 'B', 'B', 'C', 'C', 'C', 'D', 'D', 'D', 'D', 'E', 'E', 'E']
col = ['col1', 'col2', 'col3']
data = np.random.randint(0, 9, (15, 3))

# reset_index()를 사용하여 index 열을 일반 열로 변환
df = pd.DataFrame(data = data, index = idx, columns = col).reset_index()

print(df)

   index  col1  col2  col3
0      A     0     1     3
1      A     6     0     5
2      B     3     5     4
3      B     3     7     0
4      B     1     4     4
5      C     0     4     1
6      C     0     6     1
7      C     7     0     3
8      D     7     6     4
9      D     0     5     0
10     D     7     3     7
11     D     2     6     8
12     E     7     2     1
13     E     6     1     7
14     E     1     1     8


In [8]:
# groupby() 실행시 DataFrameGroupBy 객체가 반환되며, 그룹별로 연산을 수행할 수 있다.
print(df.groupby('index'))

# 추가연산 메서드 입력 : mean()
print(df.groupby('index').mean())

# 추가연산 메서드 입력 : count()
print(df.groupby('index').count())

# agg() 메서드를 사용하여 여러 연산을 동시에 수행할 수 있다.
print(df.groupby('index').agg(['mean', 'sum', 'count']))
print("agg() 이용해 여러 연산을 수행할 경우 MulitiColumns 형태로 출력")

           col1      col2      col3
index                              
A      3.000000  0.500000  4.000000
B      2.333333  5.333333  2.666667
C      2.333333  3.333333  1.666667
D      4.000000  5.000000  4.750000
E      4.666667  1.333333  5.333333
       col1  col2  col3
index                  
A         2     2     2
B         3     3     3
C         3     3     3
D         4     4     4
E         3     3     3
           col1                col2                col3          
           mean sum count      mean sum count      mean sum count
index                                                            
A      3.000000   6     2  0.500000   1     2  4.000000   8     2
B      2.333333   7     3  5.333333  16     3  2.666667   8     3
C      2.333333   7     3  3.333333  10     3  1.666667   5     3
D      4.000000  16     4  5.000000  20     4  4.750000  19     4
E      4.666667  14     3  1.333333   4     3  5.333333  16     3
agg() 이용해 여러 연산을 수행할 경우 MulitiColumns 형태로 출력


In [9]:
# group_keys인수의 사용

# top() 생성 : 상위 n개 열을 반환하는 함수
def top (df, n=2, col='col1'):
    return df.sort_values(by=col)[-n:]
print(df.groupby('index').apply(top))

          col1  col2  col3
index                     
A     0      0     1     3
      1      6     0     5
B     2      3     5     4
      3      3     7     0
C     6      0     6     1
      7      7     0     3
D     8      7     6     4
      10     7     3     7
E     13     6     1     7
      12     7     2     1


In [10]:
# index와 중복제거
print(df.groupby('index',group_keys=False).apply(top))

    col1  col2  col3
0      0     1     3
1      6     0     5
2      3     5     4
3      3     7     0
6      0     6     1
7      7     0     3
8      7     6     4
10     7     3     7
13     6     1     7
12     7     2     1


In [11]:
# observed 인수의 사용
df_cat = pd.Categorical(df['index'], categories=['A','B','C','D','E','F']) # df의 index열에 대해서 A,B,C,D,E,F 로 Categorical을 하여 df_cat 생성
print(df_cat)

['A', 'A', 'B', 'B', 'B', ..., 'D', 'D', 'E', 'E', 'E']
Length: 15
Categories (6, str): ['A', 'B', 'C', 'D', 'E', 'F']


In [12]:
# col1 을 groupby() 기준으로 지정
print(df['col1'].groupby(df_cat).count())

A    2
B    3
C    3
D    4
E    3
Name: col1, dtype: int64


In [13]:
# as_index인수의 사용
print(df.groupby(['index'],as_index=False).sum())

  index  col1  col2  col3
0     A     6     1     8
1     B     7    16     8
2     C     7    10     5
3     D    16    20    19
4     E    14     4    16


In [ ]:
# dropna인수의 사용 (dropna인수를 통해 결측값(NaN)이 포함된 경우 그룹화에서 제외할지 여부를 정할 수 있다.)

# 먼저 index열의 6번행을 결측값(NaN)으로 변경
df.loc[6, 'index'] = np.nan
print(df)

   index  col1  col2  col3
0      A     0     1     3
1      A     6     0     5
2      B     3     5     4
3      B     3     7     0
4      B     1     4     4
5      C     0     4     1
6    NaN     0     6     1
7      C     7     0     3
8      D     7     6     4
9      D     0     5     0
10     D     7     3     7
11     D     2     6     8
12     E     7     2     1
13     E     6     1     7
14     E     1     1     8


In [17]:
# 일반적인 사용시 NaN은 계산에서 제외
print(df.groupby('index').sum())

       col1  col2  col3
index                  
A         6     1     8
B         7    16     8
C         7     4     4
D        16    20    19
E        14     4    16


In [18]:
# dropna=False인 경우 인덱스에 NaN이 포함
print(df.groupby('index', dropna=False).sum())

       col1  col2  col3
index                  
A         6     1     8
B         7    16     8
C         7     4     4
D        16    20    19
E        14     4    16
NaN       0     6     1


In [19]:
# level인수의 사용 (Multi Index)
# : Multi Index의 경우 level을 숫자나 str 형태로 지정해 groupby()를 수행할 수 있다.

# Multi Index 생성
idx = [['idx1','idx1','idx2','idx2','idx2'],['row1','row2','row1','row2','row3']]
col = ['col1','col2','col2']
data = np.random.randint(0,9,(5,3))
df = pd.DataFrame(data=data, index = idx, columns = col).rename_axis(index=['lv0','lv1'])
print(df)

           col1  col2  col2
lv0  lv1                   
idx1 row1     6     2     4
     row2     2     8     8
idx2 row1     0     3     7
     row2     5     8     2
     row3     1     5     6


In [20]:
# level을 int형태로 지정하여 groupby() 수행
print(df.groupby(level=1).sum())

      col1  col2  col2
lv1                   
row1     6     5    11
row2     7    16    10
row3     1     5     6


In [24]:
# level을 str로 지정해주는경우 + (여러개 지정시 순차적으로 groupby 된다.)
print(df.groupby(['lv1','lv0']).sum())

           col1  col2  col2
lv1  lv0                   
row1 idx1     6     2     4
     idx2     0     3     7
row2 idx1     2     8     8
     idx2     5     8     2
row3 idx2     1     5     6
